In [ ]:
import os
import emcee
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

import belfast.mcmc_utils as mcmc
from belfast.target_utils import query_data
from belfast.target_definition import Target
from belfast.paths import REPO_ROOT, RESULTS_DIR, DATA_DIR

%load_ext autoreload
%autoreload 2

# Prepare data


### You may run a test using mock data generated from PARSEC V2.0 isochrones

In [ ]:
target_name =      'Mock Hyades star'
cluster =          ''
true_values = np.array([2, 700, 0.09, 0.8, np.deg2rad(30), 10])
obs_mags = np.array([4.965, 5.033, 4.837, 5.161, 5.03 , 4.699, 4.675, 4.652])
obs_err_mags = np.array([0.002,  0.002,  0.002,   0.02,    0.02,    0.02,   0.02,    0.02 ])
obs_mags = obs_mags + np.random.normal(loc=0.0, scale=[0.02]*8)

vsini_obs =        88
vsini_obs_err =    5
parallax_obs =     18.3851
parallax_obs_err = 0.1772
extinction =       0.01
extinction_err =   0.01
extinction_type =  'A_V'

# Set Target

In [ ]:
target = Target(name = target_name,
                bands_list = ['G', 'G_BP', 'G_RP', 'B_T', 'V_T', 'J', 'H', 'Ks'],
                obs_mags = obs_mags,
                obs_err_mags = obs_err_mags,
                vsini_obs = vsini_obs,
                vsini_obs_err = vsini_obs_err,
                parallax_obs = parallax_obs,
                parallax_obs_err = parallax_obs_err,
                sigma_adjustment_parameter = 1,
                extinction_parameter = True,
                true_values=False,
                )

In [ ]:
# sample all parameters
target.add_info(cluster=cluster,
                sampling_variables_list = ['Mini', 'Age', 'MH', 'omega_ini', 'Inclination', 'parallax'],
                extinction=extinction,
                extinction_err=extinction_err,
                extinction_type=extinction_type
)

# target.known_parameter(true_values=true_values) # for mock stars, you may set the true values and overplot them on corner plots

In [ ]:
# # if you want to sample with fixed metallicity [M/H] = 0.0, please give the value by target.known_parameter()
# target.add_info(cluster=cluster,
#                 sampling_variables_list = ['Mini', 'Age', 'omega_ini', 'Inclination', 'parallax'], # not sampling MH
#                 extinction=extinction,
#                 extinction_err=extinction_err,
#                 extinction_type=extinction_type
# )
#
# target.known_parameter(known_MH=0) # giving fixed MH here

In [ ]:
# # If assuming non-rotation
# target.add_info(cluster=cluster,
#                 sampling_variables_list = ['Mini', 'Age', 'MH', 'parallax'], # not sampling omega_ini and inclination
#                 extinction=extinction,
#                 extinction_err=extinction_err,
#                 extinction_type=extinction_type
# )
#
# target.known_parameter(known_omega_ini=1e-8, known_Inclination=1e-8)

# Set method

In [ ]:
# Standard Regularization + METHOD DIRECT

target.set_method(
    grids_type='Mini',
    METHOD='DIRECT',
)

In [ ]:
# # Standard Regularization + METHOD PARS
#
# target.set_method(
#     grids_type='Mini',
#     METHOD='PARS',
# )

In [ ]:
# # Reparameterized Regularization + METHOD DIRECT
#
# target.set_method(
#     grids_type='u',
#     METHOD='DIRECT',
# )

In [ ]:
# # Reparameterized Regularization + METHOD PARS
#
# target.set_method(
#     grids_type='u',
#     METHOD='PARS',
# )

In [ ]:
# # MIST isochrones + METHOD PARS
#
# target.set_method(
#     grids_type='Mini',
#     iso_name='MIST',
#     METHOD='PARS',
# )

# Sampling

In [ ]:
# Set initial states and prior distributions for sampling.

target.set_sampling(n_walkers=30,
                    inclination_prior = lambda x: norm.logpdf(x, loc=np.deg2rad(30), scale=np.deg2rad(6)),
                    extinction_prior = lambda x: norm.logpdf(x, loc=extinction, scale=extinction_err),
                    )

In [ ]:
# quick-look tests

# target.emcee_moves = [
#     (emcee.moves.StretchMove(a=2.0),         0.50),
#     (emcee.moves.DEMove(gamma0=0.40),        0.20),
#     (emcee.moves.DESnookerMove(),            0.15),
#     (emcee.moves.KDEMove(),                  0.15),
# ]

target.emcee_moves = [
    (emcee.moves.StretchMove(a=2.0),         0.95),
    (emcee.moves.KDEMove(),                  0.05),
]

# target.Vrot_prior = None
# target.Vrot_prior = lambda x: 0
mcmc.target = target

sampler_test, target = mcmc.run_test_sampling(
    # n_steps_evaluate=1000,
    n_steps_test=3000,
    n_processes=8,
    save_samples=True,
)

In [ ]:
# Run sampling!
# Set max time for running and processes for parallel

mcmc.target = target
sampler = mcmc.run_sampling(initial_steps = 10000, # the interval for checking convergence (e.g. 10000, 30000, 60000)
                            max_hours=10, # hours; if reaching max_hours, stop sampling anyway.
                            n_processes=8, # the number of threads used for parallel sampling.
                            )

# Play sounds to indicate completion. (For Mac only.)
for _ in range(3):
	os.system("afplay /System/Library/Sounds/Glass.aiff")

In [ ]:
 # Save figures and samples, including corner plot, trace plots for all walkers, target file, and summary of results.

mcmc.target = target
mcmc.save_and_plot(sampler, plot_samples=20000, folder_suffix=f'G-Tomo A_V prior')